In [1]:
"""
Bulletproof Stock Price Fetcher - GUARANTEED WORKING
====================================================

This version handles ALL edge cases and data type issues
"""

import pandas as pd
import numpy as np
from typing import List, Dict
import warnings
warnings.filterwarnings('ignore')

try:
    import yfinance as yf
except ImportError:
    print("⚠ Install yfinance: pip install yfinance")
    yf = None


class BulletproofPriceFetcher:
    """
    Bulletproof version that handles all data type conversions correctly
    """
    
    def __init__(self):
        if yf is None:
            raise ImportError("yfinance not installed. Install with: pip install yfinance")
    
    def download_ticker_data(self, ticker: str, start_year: int, end_year: int) -> pd.DataFrame:
        """
        Download data for a ticker across multiple years
        
        Returns clean DataFrame with proper column names
        """
        print(f"  Downloading {ticker} ({start_year}-{end_year})...", end='')
        
        all_data = []
        
        for year in range(start_year, end_year + 1):
            try:
                data = yf.download(
                    ticker,
                    start=f'{year}-01-01',
                    end=f'{year}-12-31',
                    progress=False
                )
                
                if not data.empty:
                    all_data.append(data)
            
            except Exception as e:
                print(f"\n    ⚠ Error for {year}: {e}")
                continue
        
        if not all_data:
            print(" ✗ No data")
            return pd.DataFrame()
        
        # Combine all years
        combined = pd.concat(all_data)
        
        # Reset index to make Date a column (IMPORTANT!)
        combined = combined.reset_index()
        #print( combined.columns)
        
        if isinstance(combined.columns, pd.MultiIndex):
            combined.columns = combined.columns.get_level_values(0)
            
        # Now safe to lowercase
        combined.columns = [col.lower().strip() for col in combined.columns]

        #print( combined.columns ) 

        # Add ticker column
        combined['ticker'] = ticker
        
        print(f" ✓ ({len(combined)} days)")
        
        return combined
    
    def get_price_for_date(self, data: pd.DataFrame, ticker: str, target_date: str) -> Dict:
        """
        Extract price for a specific date
        
        Parameters:
        -----------
        data : pd.DataFrame
            Data from download_ticker_data()
        ticker : str
            Stock ticker
        target_date : str
            Target date in 'YYYY-MM-DD' format
        
        Returns:
        --------
        dict with price and date info
        """
        if data.empty:
            return {
                'ticker': ticker,
                'requested_date': target_date,
                'actual_date': None,
                'close_price': np.nan,
                'days_diff': None,
                'found': False
            }
        
        # Ensure date columns are datetime
        data['date'] = pd.to_datetime(data['date'])
        target_dt = pd.to_datetime(target_date)
        
        # Calculate differences
        data['days_diff'] = abs((data['date'] - target_dt).dt.days)
        
        # Find minimum
        min_idx = data['days_diff'].idxmin()
        
        # Get the row (IMPORTANT: use .loc with explicit column names)
        closest_date = data.loc[min_idx, 'date']
        closest_price = data.loc[min_idx, 'close']
        closest_diff = data.loc[min_idx, 'days_diff']
        
        # Convert date to string (BULLETPROOF method)
        if pd.isna(closest_date):
            actual_date_str = None
        else:
            # Use .to_pydatetime() for absolute safety
            actual_date_str = closest_date.strftime('%Y-%m-%d')
        
        return {
            'ticker': ticker,
            'requested_date': target_date,
            'actual_date': actual_date_str,
            'close_price': float(closest_price),
            'days_diff': int(closest_diff),
            'found': True
        }
    
    def extract_prices_for_multiple_dates(self, ticker: str, dates: List[str], 
                                         start_year: int, end_year: int) -> pd.DataFrame:
        """
        MAIN METHOD: Extract prices for multiple dates
        
        This is the bulletproof version that handles all edge cases
        """
        # Download data once
        data = self.download_ticker_data(ticker, start_year, end_year)
        
        if data.empty:
            print(f"  ⚠ No data for {ticker}")
            return pd.DataFrame()
        
        # Extract prices for each date
        results = []
        found_count = 0
        
        for target_date in dates:
            result = self.get_price_for_date(data, ticker, target_date)
            results.append(result)
            
            if result['found']:
                found_count += 1
        
        result_df = pd.DataFrame(results)
        print(f"  ✓ Found {found_count}/{len(dates)} prices")
        
        return result_df


# ==================== SIMPLE USAGE FUNCTION ====================

def fetch_and_merge_prices(fundamentals_df: pd.DataFrame, 
                          tickers: List[str],
                          start_year: int, 
                          end_year: int) -> pd.DataFrame:
    """
    All-in-one function to fetch prices and merge with fundamentals
    
    Parameters:
    -----------
    fundamentals_df : pd.DataFrame
        Your fundamental data with 'ticker' and 'fiscalDateEnding' columns
    tickers : list
        List of stock tickers
    start_year : int
        Start year for price download
    end_year : int
        End year for price download
    
    Returns:
    --------
    pd.DataFrame : Fundamentals with prices merged
    """
    
    print("="*80)
    print("FETCHING STOCK PRICES FOR FISCAL DATES")
    print("="*80)
    
    # Initialize fetcher
    fetcher = BulletproofPriceFetcher()
    
    # Get unique fiscal dates
    fiscal_dates = fundamentals_df['fiscalDateEnding'].unique().tolist()
  
    fundamentals_df['fiscalDateEnding'] = pd.to_datetime(fundamentals_df['fiscalDateEnding'])
    start_year = int(fundamentals_df['fiscalDateEnding'].dt.year.min())
    end_year = int(fundamentals_df['fiscalDateEnding'].dt.year.max())

    print(f"Year range: {start_year} to {end_year}")
    print(f"\nFetching prices for {len(tickers)} tickers and {len(fiscal_dates)} dates...")
    print(f"Date range: {start_year}-01-01 to {end_year}-12-31\n")
    
    # Fetch prices for each ticker
    all_prices = []
    
    for ticker in tickers:
        prices = fetcher.extract_prices_for_multiple_dates( ticker=ticker,
                                                            dates=fiscal_dates,
                                                            start_year=start_year,
                                                            end_year=end_year
                                                          )
        
        if not prices.empty:
            all_prices.append(prices)
    
    if not all_prices:
        print("⚠ No prices found!")
        return fundamentals_df
    
    # Combine all prices
    prices_df = pd.concat(all_prices, ignore_index=True)
    
    print(f"\n✓ Total prices fetched: {len(prices_df)}")
    print(f"✓ Prices found: {prices_df['found'].sum()}/{len(prices_df)}")
    
    # Merge with fundamentals
    print("\nMerging with fundamentals...")
    
    # Rename column for merge
    prices_df_merge = prices_df[['ticker', 'requested_date', 'close_price', 'actual_date']].copy()
    prices_df_merge.columns = ['ticker', 'fiscalDateEnding', 'close_price', 'price_date']

    fundamentals_df['fiscalDateEnding'] = pd.to_datetime(fundamentals_df['fiscalDateEnding'])
    prices_df_merge['fiscalDateEnding'] = pd.to_datetime(prices_df_merge['fiscalDateEnding'])

    # Merge
    merged = fundamentals_df.merge(
        prices_df_merge,
        on=['ticker', 'fiscalDateEnding'],
        how='left'
    )
    
    prices_added = merged['close_price'].notna().sum()
    print(f"✓ Added prices to {prices_added}/{len(merged)} records")
    
    return merged


# ==================== ULTRA-SIMPLE VERSION ====================

def ultra_simple_fetch(tickers: List[str], start_date: str, end_date: str) -> pd.DataFrame:
    """
    Ultra-simple: Just download prices, no fancy stuff
    
    Parameters:
    -----------
    tickers : list
        ['AAPL', 'MSFT', 'GOOGL']
    start_date : str
        '2021-01-01'
    end_date : str
        '2023-12-31'
    
    Returns:
    --------
    pd.DataFrame with Date, ticker, Close columns
    """
    print(f"Downloading {len(tickers)} tickers from {start_date} to {end_date}...")
    
    all_data = []
    
    for ticker in tickers:
        print(f"  {ticker}...", end='')
        
        try:
            data = yf.download(ticker, start=start_date, end=end_date, progress=False)
            
            if not data.empty:
                data = data.reset_index()
                data['ticker'] = ticker
                all_data.append(data)
                print(" ✓")
            else:
                print(" ✗ No data")
        
        except Exception as e:
            print(f" ✗ Error: {e}")
    
    if all_data:
        result = pd.concat(all_data, ignore_index=True)
        print(f"\n✓ Downloaded {len(result)} price records")
        return result
    
    return pd.DataFrame()


def find_closest_price(prices_df: pd.DataFrame, ticker: str, target_date: str) -> Dict:
    """
    Find closest price for a ticker on a date
    
    Ultra-safe version that won't have type errors
    """
    # Filter by ticker
    ticker_data = prices_df[prices_df['ticker'] == ticker].copy()
    
    if ticker_data.empty:
        return {'price': np.nan, 'actual_date': None, 'found': False}
    
    # Ensure date is datetime
    ticker_data['Date'] = pd.to_datetime(ticker_data['Date'])
    target = pd.to_datetime(target_date)
    
    # Find closest
    ticker_data['diff'] = abs((ticker_data['Date'] - target).dt.days)
    min_idx = ticker_data['diff'].idxmin()
    
    # Get values using .loc (BULLETPROOF)
    actual_date = ticker_data.loc[min_idx, 'Date']
    close_price = ticker_data.loc[min_idx, 'Close']
    
    # Convert date safely
    if pd.isna(actual_date):
        date_str = None
    else:
        date_str = actual_date.strftime('%Y-%m-%d')
    
    return {
        'target_date': target_date,
        'actual_date': date_str,
        'price': float(close_price),
        'found': True
    }


# ==================== COMPLETE WORKING EXAMPLE ====================

if __name__ == "__main__":
    
    print("\n" + "█"*80)
    print("█" + " "*20 + "BULLETPROOF PRICE FETCHER" + " "*34 + "█")
    print("█"*80)
    
    # OPTION 1: Ultra-simple (just download prices)
    print("\n" + "="*80)
    print("OPTION 1: ULTRA-SIMPLE - Just download prices")
    print("="*80)
    
    code1 = '''
import pandas as pd
from bulletproof_fetcher import ultra_simple_fetch, find_closest_price

# Download prices
prices = ultra_simple_fetch(
    tickers=['AAPL', 'MSFT'],
    start_date='2023-09-01',
    end_date='2023-09-30'
)

print(prices.head())

# Find price for specific date
result = find_closest_price(prices, 'AAPL', '2023-09-30')
print(f"AAPL on {result['target_date']}: ${result['price']:.2f}")
'''
    
    print(code1)
    
    # OPTION 2: With fundamentals merge
    print("\n" + "="*80)
    print("OPTION 2: WITH FUNDAMENTALS - Complete workflow")
    print("="*80)
    
    code2 = '''
import pandas as pd
from bulletproof_fetcher import fetch_and_merge_prices

# Load your fundamentals
df_fundamentals = pd.read_csv('fundamentals.csv')

# Fetch prices and merge
df_merged = fetch_and_merge_prices(
    fundamentals_df=df_fundamentals,
    tickers=df_fundamentals['ticker'].unique().tolist(),
    start_year=2021,
    end_year=2023
)

# Save
df_merged.to_csv('fundamentals_with_prices.csv', index=False)
print("✓ Done!")
'''
    
    print(code2)
    
    # Test
    print("\n" + "="*80)
    print("QUICK TEST")
    print("="*80)
    
    print("\nDownloading sample data...")
    
    try:
        prices = ultra_simple_fetch(['AAPL', 'MSFT'], '2023-09-01', '2023-09-30')
        
        if not prices.empty:
            print("\n✓ Sample data:")
            print(prices[['Date', 'ticker', 'Close']].head())
            
            result = find_closest_price(prices, 'AAPL', '2023-09-30')
            print(f"\n✓ Test result:")
            print(f"  Requested: 2023-09-30")
            print(f"  Actual: {result['actual_date']}")
            print(f"  Price: ${result['price']:.2f}")
    
    except Exception as e:
        print(f"⚠ Error: {e}")


████████████████████████████████████████████████████████████████████████████████
█                    BULLETPROOF PRICE FETCHER                                  █
████████████████████████████████████████████████████████████████████████████████

OPTION 1: ULTRA-SIMPLE - Just download prices

import pandas as pd
from bulletproof_fetcher import ultra_simple_fetch, find_closest_price

# Download prices
prices = ultra_simple_fetch(
    tickers=['AAPL', 'MSFT'],
    start_date='2023-09-01',
    end_date='2023-09-30'
)

print(prices.head())

# Find price for specific date
result = find_closest_price(prices, 'AAPL', '2023-09-30')
print(f"AAPL on {result['target_date']}: ${result['price']:.2f}")


OPTION 2: WITH FUNDAMENTALS - Complete workflow

import pandas as pd
from bulletproof_fetcher import fetch_and_merge_prices

# Load your fundamentals
df_fundamentals = pd.read_csv('fundamentals.csv')

# Fetch prices and merge
df_merged = fetch_and_merge_prices(
    fundamentals_df=df_fundamentals,
  

In [2]:
df_fundamentals = pd.read_csv('financial_features.csv')

# Fetch prices and merge
df_merged = fetch_and_merge_prices( fundamentals_df=df_fundamentals,
                                    tickers=df_fundamentals['ticker'].unique().tolist(),
    start_year=2021,
    end_year=2023
)

# Save
df_merged.to_csv('fundamentals_with_prices.csv', index=False)
print("✓ Done!")

FETCHING STOCK PRICES FOR FISCAL DATES
Year range: 2020 to 2026

Fetching prices for 58 tickers and 72 dates...
Date range: 2020-01-01 to 2026-12-31

  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ANSS"}}}

1 Failed download:
['ANSS']: YFTzMissingError('possibly delisted; no timezone found')
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ANSS"}}}

1 Failed download:
['ANSS']: YFTzMissingError('possibly delisted; no timezone found')

1 Failed download:
['ANSS']: YFTzMissingError('possibly delisted; no timezone found')

1 Failed download:
['ANSS']: YFTzMissingError('possibly delisted; no timezone found')

1 Failed download:
['ANSS']: YFTzMissingError('possibly delisted; no timezone found')

1 Failed download:
['ANSS']: YFTzMissingError('possibly delisted; no timezone found')

1 Failed download:
['ANSS']: YFTzMissingError('possibly delisted; no timezone found')


 ✗ No data
  ⚠ No data for ANSS
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices


1 Failed download:
['SGEN']: YFTzMissingError('possibly delisted; no timezone found')

1 Failed download:
['SGEN']: YFTzMissingError('possibly delisted; no timezone found')

1 Failed download:
['SGEN']: YFTzMissingError('possibly delisted; no timezone found')

1 Failed download:
['SGEN']: YFTzMissingError('possibly delisted; no timezone found')

1 Failed download:
['SGEN']: YFTzMissingError('possibly delisted; no timezone found')

1 Failed download:
['SGEN']: YFTzMissingError('possibly delisted; no timezone found')

1 Failed download:
['SGEN']: YFTzMissingError('possibly delisted; no timezone found')


 ✗ No data
  ⚠ No data for SGEN


1 Failed download:
['SPLK']: YFTzMissingError('possibly delisted; no timezone found')

1 Failed download:
['SPLK']: YFTzMissingError('possibly delisted; no timezone found')

1 Failed download:
['SPLK']: YFTzMissingError('possibly delisted; no timezone found')

1 Failed download:
['SPLK']: YFTzMissingError('possibly delisted; no timezone found')

1 Failed download:
['SPLK']: YFTzMissingError('possibly delisted; no timezone found')

1 Failed download:
['SPLK']: YFTzMissingError('possibly delisted; no timezone found')

1 Failed download:
['SPLK']: YFTzMissingError('possibly delisted; no timezone found')


 ✗ No data
  ⚠ No data for SPLK
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices
  ✓ Found 72/72 prices

✓ Total prices fetched: 3960
✓ Prices found: 3960/3960

Merging with fundamentals...
✓ Added prices to 1308/1360 records
✓ Done!


In [3]:
df_merged.columns

Index(['ticker', 'fiscalDateEnding', 'current_ratio', 'quick_ratio',
       'cash_ratio', 'working_capital_ratio', 'gross_profit_margin',
       'operating_profit_margin', 'net_profit_margin', 'ebitda_margin', 'roa',
       'roe', 'debt_to_equity', 'debt_to_assets', 'equity_ratio',
       'interest_coverage', 'asset_turnover', 'receivables_turnover',
       'inventory_turnover', 'days_inventory_outstanding',
       'days_sales_outstanding', 'operating_expense_ratio', 'sga_ratio',
       'rd_intensity', 'quality_of_earnings', 'depreciation_to_ppe',
       'cost_of_revenue_ratio', 'tax_rate', 'ebit_to_revenue',
       'intangible_asset_ratio', 'goodwill_ratio', 'tangible_asset_ratio',
       'long_term_debt_ratio', 'short_term_debt_ratio', 'debt_composition',
       'log_assets', 'share_growth', 'totalAssets_was_missing',
       'totalCurrentAssets_was_missing',
       'cashAndCashEquivalentsAtCarryingValue_was_missing',
       'cashAndShortTermInvestments_was_missing', 'inventory_was_mi

In [4]:
df_merged.head(10)

,ticker,fiscalDateEnding,current_ratio,quick_ratio,cash_ratio,working_capital_ratio,gross_profit_margin,operating_profit_margin,net_profit_margin,ebitda_margin,...,interestExpense_was_missing,depreciationAndAmortization_was_missing,incomeBeforeTax_was_missing,incomeTaxExpense_was_missing,netIncomeFromContinuingOperations_was_missing,ebit_was_missing,ebitda_was_missing,netIncome_was_missing,close_price,price_date
0,INTC,2025-12-31,2.017039,1.649089,1.184988,0.151886,0.361489,0.040222,-0.043221,0.266784,...,0,0,0,0,0,0,0,0,37.299999,2025-12-30
1,INTC,2025-09-30,1.601728,1.245998,0.957829,0.095025,0.382187,0.050026,0.297590,0.574819,...,0,0,0,0,0,0,0,0,33.549999,2025-09-30
2,INTC,2025-06-30,1.240491,0.915118,0.606475,0.043679,0.275449,-0.246987,-0.226923,0.036628,...,0,0,0,0,0,0,0,0,22.400000,2025-06-30
3,INTC,2025-03-31,1.309567,0.927861,0.654193,0.051810,0.368832,-0.023763,-0.064814,0.188442,...,0,0,0,0,0,0,0,0,22.709999,2025-03-31
4,INTC,2024-12-31,1.326866,0.984860,0.618572,0.059333,0.391585,0.028892,-0.008836,0.244039,...,0,0,0,0,0,0,0,0,19.820000,2024-12-30
5,INTC,2024-09-30,1.312239,0.969169,0.685059,0.056722,0.150331,-0.681798,-1.252559,-0.393406,...,0,0,0,0,0,0,0,0,23.459999,2024-09-30
6,INTC,2024-06-30,1.587067,1.235988,0.914010,0.091181,0.354321,-0.153043,-0.125458,0.066937,...,0,0,0,0,0,0,0,0,30.645597,2024-07-01
7,INTC,2024-03-31,1.565722,1.143351,0.783118,0.079877,0.410013,-0.084014,-0.029943,0.164257,...,0,0,0,0,0,0,0,0,44.060402,2024-04-01
8,INTC,2023-12-31,1.542402,1.145760,0.892382,0.079427,0.457419,0.167792,0.173244,0.361418,...,0,0,0,0,0,0,0,0,49.585903,2023-12-29
9,INTC,2023-09-30,1.531104,1.130391,0.874747,0.080477,0.425060,-0.000565,0.020978,0.183571,...,0,0,0,0,0,0,0,0,34.965206,2023-09-29
